# CWRU Bearing — Notebook 3: Preprocessing

**Prerequisite:** Run `01_download.ipynb` and `02_eda_clean.ipynb` first.

This notebook covers every step between raw signal and model-ready input:

| Step | Description |
|------|-------------|
| 1 | Load raw signals from manifest |
| 2 | Sliding window segmentation |
| 3 | Z-score normalisation (per window) |
| 4 | Label assignment |
| 5 | Train / Val / Test split (load-based, no leakage) |
| 6 | Feature extraction → classical ML path (RQ1) |
| 7 | Model input preparation → deep learning path (RQ2–4) |
| 8 | Save all outputs for Notebook 4 |

**Outputs saved:**
- `preprocessed/X_train_dl.npy`, `X_val_dl.npy`, `X_test_dl.npy` — DL tensors
- `preprocessed/y_train.npy`, `y_val.npy`, `y_test.npy` — labels
- `preprocessed/F_train.npy`, `F_val.npy`, `F_test.npy` — classical features
- `preprocessed/class_info.json` — label names, class weights, split summary

---
## Step 0 — Imports and configuration

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.io import loadmat
from scipy.fft import fft, fftfreq
from scipy.stats import kurtosis as kurt_stat, skew as skew_stat
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import StandardScaler
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
np.random.seed(42)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_DIR   = 'cwru_data'
OUT_DIR    = 'preprocessed'
PLOTS_DIR  = 'preprocessing_plots'
os.makedirs(OUT_DIR,   exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

# ── Window parameters ────────────────────────────────────────────────────────
# 1024 samples chosen because:
#   FFT resolution = 12000 / 1024 = 11.7 Hz
#   Minimum gap between BPFO(107.4), BSF(141.2), BPFI(162.2) = 21 Hz
#   N=512 gives Δf=23.4Hz which cannot resolve BSF from BPFI → too small
#   N=1024 gives Δf=11.7Hz → all three frequencies resolved
WINDOW  = 1024
OVERLAP = 0.5
STEP    = int(WINDOW * (1 - OVERLAP))   # = 512 samples
FS      = 12000   # sampling rate (Hz)

# ── Bearing characteristic frequencies (SKF 6205-2RS) ────────────────────────
# Derived from: frequency_multiple × (RPM / 60)
# Multiples from CWRU bearing specification sheet
MULTIPLES = {'BPFO': 3.5848, 'BPFI': 5.4152, 'BSF': 4.7135, 'FTF': 0.39828}
RPM_BY_LOAD = {0: 1797, 1: 1772, 2: 1750, 3: 1730}

def bearing_freqs(rpm):
    hz = rpm / 60
    return {k: round(v * hz, 2) for k, v in MULTIPLES.items()}

# ── Split definition ─────────────────────────────────────────────────────────
# Split at FILE level (not segment level) to prevent data leakage.
# Segments from the same signal file must all land in the same partition.
TRAIN_LOADS = [0, 2]   # loads used for training
VAL_LOADS   = [1]      # held out for hyperparameter tuning
TEST_LOADS  = [3]      # never seen during training — RQ4 domain shift test

print('Configuration loaded.')
print(f'Window: {WINDOW} samples = {WINDOW/FS*1000:.1f} ms at {FS} S/s')
print(f'Step:   {STEP} samples  (50% overlap)')
print(f'FFT resolution: {FS/WINDOW:.1f} Hz')
print()
print('Bearing frequencies at each load:')
for load, rpm in RPM_BY_LOAD.items():
    f = bearing_freqs(rpm)
    print(f'  Load {load}HP ({rpm}RPM): BPFO={f["BPFO"]}  BSF={f["BSF"]}  BPFI={f["BPFI"]}  FTF={f["FTF"]}')

---
## Step 1 — Load raw signals from manifest

In [ ]:
df_manifest = pd.read_csv('cwru_manifest.csv')
df_manifest = df_manifest[df_manifest['present'] == True].reset_index(drop=True)

# Use 12k drive-end only for main experiments
df_12k = df_manifest[df_manifest['sample_rate'] == 12000].reset_index(drop=True)
print(f'12k files in manifest: {len(df_12k)}')

def extract_de_signal(fpath):
    """Extract drive-end (DE_time) channel from a CWRU .mat file."""
    mat = loadmat(fpath)
    key = next((k for k in mat if 'DE_time' in k), None)
    if key is None:
        key = next((k for k in mat if not k.startswith('_')), None)
    return mat[key].flatten().astype(np.float32)

records = []
for _, row in df_12k.iterrows():
    fpath = os.path.join(DATA_DIR, row['filename'])
    sig   = extract_de_signal(fpath)
    records.append({
        'filename':   row['filename'],
        'fault_type': row['fault_type'],
        'fault_loc':  row['fault_loc'],
        'severity':   row['severity'],
        'load_hp':    row['load_hp'],
        'rpm':        row['rpm'],
        'signal':     sig,
        'sig_len':    len(sig)
    })

print(f'Loaded {len(records)} signal files.')
print(f'Total raw samples: {sum(r["sig_len"] for r in records):,}')

---
## Step 2 — Sliding window segmentation

Each ~10-second signal is sliced into 1024-sample windows with 50% overlap.
A window of 1024 samples at 12k S/s = **85.3 ms** — long enough to capture
multiple cycles of BPFO (period = 9.3ms → ~9 cycles per window).

In [ ]:
def slide_windows(signal, window, step):
    """
    Slice a 1-D signal into overlapping windows.
    Returns list of numpy arrays, each of length `window`.
    """
    windows = []
    for start in range(0, len(signal) - window + 1, step):
        windows.append(signal[start : start + window].copy())
    return windows

# Quick preview — how many windows does one file produce?
sample_rec = records[0]
sample_wins = slide_windows(sample_rec['signal'], WINDOW, STEP)
print(f"File: {sample_rec['filename']}")
print(f"  Signal length : {sample_rec['sig_len']:,} samples")
print(f"  Windows (raw) : {len(sample_wins)}")
print(f"  Window shape  : {sample_wins[0].shape}")
print(f"  Window duration: {WINDOW/FS*1000:.1f} ms")
print()

# Visualise the segmentation concept
fig, axes = plt.subplots(2, 1, figsize=(14, 6))
t_full = np.arange(len(sample_rec['signal'])) / FS
axes[0].plot(t_full[:4096], sample_rec['signal'][:4096], linewidth=0.6, color='steelblue')
axes[0].set_title('Full signal (first 341ms shown)', fontweight='bold')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude (V)')

# Show first 4 windows with their boundaries
colors = ['#E65100','#1565C0','#2E7D32','#6A1B9A']
t_win  = np.arange(WINDOW) / FS
for i, col in enumerate(colors):
    start = i * STEP
    win   = sample_rec['signal'][start:start+WINDOW]
    axes[1].plot(t_win + start/FS, win, linewidth=0.8, color=col,
                 label=f'Window {i+1} (start={start/FS*1000:.0f}ms)')
    axes[1].axvspan(start/FS, (start+WINDOW)/FS, alpha=0.06, color=col)

axes[1].set_title(f'First 4 windows ({WINDOW} samples, {STEP}-sample step = 50% overlap)', fontweight='bold')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Amplitude (V)')
axes[1].legend(fontsize=9)

plt.suptitle(f'Sliding window segmentation — {sample_rec["filename"]}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/step2_segmentation.png', bbox_inches='tight')
plt.show()
print('Saved: step2_segmentation.png')

---
## Step 3 — Z-score normalisation (per window)

Each window is independently normalised: **subtract mean, divide by std**.

Purpose:
- Removes the DC offset (tiny non-zero mean inherent to the sensor)
- Removes amplitude differences between motor loads
  (a 3HP signal is slightly louder than 0HP — we want the model to learn *fault shape*, not load level)
- Keeps impulse patterns (kurtosis, spectral shape) intact

In [ ]:
def zscore_normalise(window):
    """
    Z-score normalise a single window.
    Formula: (x - mean) / (std + epsilon)
    epsilon = 1e-8 prevents division by zero on silent/near-flat windows.
    """
    mean = window.mean()
    std  = window.std()
    return (window - mean) / (std + 1e-8)

# ── Visual: before vs after normalisation ────────────────────────────────────
# Compare two windows: Normal (small amplitude) vs OR@6_21 (large amplitude)
rec_normal = next(r for r in records if r['fault_type']=='Normal' and r['load_hp']==0)
rec_fault  = next(r for r in records if r['fault_loc']=='OR@6' and r['severity']==21 and r['load_hp']==3)

w_normal_raw = rec_normal['signal'][:WINDOW]
w_fault_raw  = rec_fault['signal'][:WINDOW]
w_normal_norm = zscore_normalise(w_normal_raw)
w_fault_norm  = zscore_normalise(w_fault_raw)

t = np.arange(WINDOW) / FS * 1000

fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharey=False)

# Raw
axes[0][0].plot(t, w_normal_raw, color='#607D8B', linewidth=0.7)
axes[0][0].set_title(f'Normal — RAW\nMean={w_normal_raw.mean():.4f}V  Std={w_normal_raw.std():.4f}V', fontsize=10)
axes[0][0].set_ylabel('Amplitude (V)')

axes[0][1].plot(t, w_fault_raw, color='#E65100', linewidth=0.7)
axes[0][1].set_title(f'OR@6_21 Load3HP — RAW\nMean={w_fault_raw.mean():.4f}V  Std={w_fault_raw.std():.4f}V', fontsize=10)

# Normalised
axes[1][0].plot(t, w_normal_norm, color='#607D8B', linewidth=0.7)
axes[1][0].set_title(f'Normal — Z-SCORE NORMALISED\nMean≈{w_normal_norm.mean():.4f}  Std≈{w_normal_norm.std():.4f}', fontsize=10)
axes[1][0].set_ylabel('Amplitude (std devs)')
axes[1][0].set_xlabel('Time (ms)')

axes[1][1].plot(t, w_fault_norm, color='#E65100', linewidth=0.7)
axes[1][1].set_title(f'OR@6_21 Load3HP — Z-SCORE NORMALISED\nMean≈{w_fault_norm.mean():.4f}  Std≈{w_fault_norm.std():.4f}', fontsize=10)
axes[1][1].set_xlabel('Time (ms)')

for ax_row in axes:
    for ax in ax_row:
        ax.grid(True, alpha=0.25)

plt.suptitle('Z-score normalisation: before vs after\nNote: impulse SHAPE is preserved; only scale changes', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/step3_normalisation.png', bbox_inches='tight')
plt.show()

print(f'Before normalisation:')
print(f'  Normal raw    : min={w_normal_raw.min():.4f}  max={w_normal_raw.max():.4f}  std={w_normal_raw.std():.4f} V')
print(f'  OR@6_21 raw   : min={w_fault_raw.min():.4f}  max={w_fault_raw.max():.4f}  std={w_fault_raw.std():.4f} V')
print(f'After normalisation (both have std=1, mean=0):')
print(f'  Normal normed : min={w_normal_norm.min():.2f}  max={w_normal_norm.max():.2f}')
print(f'  OR@6_21 normed: min={w_fault_norm.min():.2f}  max={w_fault_norm.max():.2f}')

---
## Step 4 — Label assignment

Every segment inherits its label from the source file's metadata.
Label format: `FaultLoc_SeverityMils` (e.g. `IR_14`, `OR@6_7`, `Ball_21`).
Normal class uses `Normal` regardless of load.

In [ ]:
# ── Define all 16 classes ─────────────────────────────────────────────────────
# Order matters — index becomes the integer class label fed to the model.
ALL_LABELS = [
    'Normal',
    'IR_7',  'IR_14',  'IR_21',  'IR_28',
    'Ball_7','Ball_14','Ball_21','Ball_28',
    'OR@6_7','OR@6_14','OR@6_21',
    'OR@3_7','OR@3_21',
    'OR@12_7','OR@12_21'
]
label2idx = {l: i for i, l in enumerate(ALL_LABELS)}
NUM_CLASSES = len(ALL_LABELS)

def assign_label(record):
    """
    Derive the class label from a record's metadata.
    Normal files get label 'Normal'.
    Fault files get 'FaultLoc_Severity' e.g. 'IR_14', 'OR@6_7'.
    """
    if record['fault_type'] == 'Normal':
        return 'Normal'
    return f"{record['fault_loc']}_{int(record['severity'])}"

# Preview label assignment on a few records
print('Label assignment preview:')
print(f'{"Filename":<30} {"Fault Loc":<10} {"Severity":<10} {"Label":<14} {"Index"}')
print('-' * 75)
for r in records[:6]:
    lbl = assign_label(r)
    idx = label2idx.get(lbl, -1)
    print(f"{r['filename']:<30} {r['fault_loc']:<10} {str(r['severity']):<10} {lbl:<14} {idx}")

print(f'\nTotal classes: {NUM_CLASSES}')
print('All labels:', ALL_LABELS)

---
## Step 5 — Train / Val / Test split (load-based, no leakage)

**Critical design decision:** Split at the FILE level, not segment level.

All segments from the same source file must land in the same partition.
Splitting by segment would allow windows from the same 10-second recording
to appear in both train and test — the model would memorise the signal
rather than learn fault patterns, producing artificially inflated accuracy.

Strategy: split by motor load — each load is a completely independent recording session:
- **Train**: loads 0 and 2 HP
- **Val**:   load 1 HP (hyperparameter selection)
- **Test**:  load 3 HP (never seen during training)

In [ ]:
def build_split(records, load_list, label2idx, window, step):
    """
    Segment all records whose load_hp is in load_list.
    Returns:
        X_raw  : (N, window)  float32 — raw (un-normalised) segments
        X_norm : (N, window)  float32 — z-score normalised segments
        y      : (N,)         int32   — class indices
        meta   : DataFrame    — per-segment metadata for analysis
    """
    X_raw, X_norm, y, meta = [], [], [], []

    for r in records:
        if r['load_hp'] not in load_list:
            continue
        lbl = assign_label(r)
        if lbl not in label2idx:
            continue
        cls = label2idx[lbl]
        wins = slide_windows(r['signal'], window, step)

        for w in wins:
            X_raw.append(w.astype(np.float32))
            X_norm.append(zscore_normalise(w).astype(np.float32))
            y.append(cls)
            meta.append({
                'label':     lbl,
                'class_idx': cls,
                'load_hp':   r['load_hp'],
                'fault_loc': r['fault_loc'],
                'severity':  r['severity'],
                'filename':  r['filename']
            })

    return (np.array(X_raw),
            np.array(X_norm),
            np.array(y, dtype=np.int32),
            pd.DataFrame(meta))

print('Building splits...')
X_raw_train, X_norm_train, y_train, meta_train = build_split(records, TRAIN_LOADS, label2idx, WINDOW, STEP)
X_raw_val,   X_norm_val,   y_val,   meta_val   = build_split(records, VAL_LOADS,   label2idx, WINDOW, STEP)
X_raw_test,  X_norm_test,  y_test,  meta_test  = build_split(records, TEST_LOADS,  label2idx, WINDOW, STEP)

print(f'\nSplit summary:')
print(f'  Train (loads {TRAIN_LOADS}): {X_norm_train.shape[0]:,} segments')
print(f'  Val   (loads {VAL_LOADS}):   {X_norm_val.shape[0]:,} segments')
print(f'  Test  (loads {TEST_LOADS}):   {X_norm_test.shape[0]:,} segments')
print(f'  Total:            {X_norm_train.shape[0]+X_norm_val.shape[0]+X_norm_test.shape[0]:,} segments')

# Verify no file appears in multiple splits
train_files = set(meta_train['filename'])
val_files   = set(meta_val['filename'])
test_files  = set(meta_test['filename'])
assert len(train_files & val_files)  == 0, 'LEAKAGE: train/val overlap'
assert len(train_files & test_files) == 0, 'LEAKAGE: train/test overlap'
assert len(val_files   & test_files) == 0, 'LEAKAGE: val/test overlap'
print('\nLeakage check PASSED — no file appears in more than one split.')

In [ ]:
# ── Visualise split class distribution ───────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
split_data = [
    (meta_train, y_train, 'Train (loads 0+2HP)', '#1565C0'),
    (meta_val,   y_val,   'Val   (load 1HP)',    '#2E7D32'),
    (meta_test,  y_test,  'Test  (load 3HP)',    '#C62828'),
]

for ax, (meta, y, title, col) in zip(axes, split_data):
    counts = pd.Series(y).map({i: l for i, l in enumerate(ALL_LABELS)}).value_counts()
    counts = counts.reindex(ALL_LABELS).fillna(0)
    ax.barh(ALL_LABELS[::-1], counts.values[::-1], color=col, alpha=0.8, edgecolor='white', linewidth=0.4)
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_xlabel('Segment count')
    ax.grid(True, alpha=0.3, axis='x')
    for i, v in enumerate(counts.values[::-1]):
        ax.text(v + 5, i, str(int(v)), va='center', fontsize=7.5)

plt.suptitle('Class distribution per split\n(uniform bars confirm no class is missing from any split)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/step5_split_distribution.png', bbox_inches='tight')
plt.show()
print('Saved: step5_split_distribution.png')

---
## Step 6 — Feature extraction (classical ML path — RQ1)

For SVM and Random Forest, each window is reduced to **14 scalar features**
rather than passing 1024 raw samples.

Features split into two groups:
- **7 time-domain**: capture amplitude and impulsiveness properties
- **7 frequency-domain**: capture energy at fault-specific frequencies

All features computed on the **raw (non-normalised)** window so that
amplitude-based features (RMS, Peak-Peak) retain their physical meaning.

In [ ]:
FEATURE_NAMES = [
    # Time-domain (7)
    'RMS',           # root mean square — signal energy
    'Kurtosis',      # 4th moment — impulse sharpness
    'Skewness',      # 3rd moment — impulse asymmetry
    'CrestFactor',   # peak / RMS — impulsiveness relative to energy
    'PeakPeak',      # max - min — total amplitude range
    'MeanAbs',       # mean absolute value
    'Energy',        # sum of squares
    # Frequency-domain (7)
    'DomFreq',       # frequency of highest FFT peak
    'FFTPeak',       # amplitude of highest FFT peak
    'BPFO_Energy',   # spectral energy in ±10Hz band around BPFO
    'BPFI_Energy',   # spectral energy in ±10Hz band around BPFI
    'BSF_Energy',    # spectral energy in ±10Hz band around BSF
    'SpectralCentroid',  # amplitude-weighted mean frequency
    'SpectralEntropy',   # frequency distribution uniformity
]

def extract_features(window, fs=12000, rpm=1797):
    """
    Extract 14 features from a raw (un-normalised) 1024-sample window.
    Uses the actual RPM to compute correct bearing characteristic frequencies.
    """
    s    = window - window.mean()         # remove DC offset
    rms  = np.sqrt(np.mean(s**2))
    peak = np.max(np.abs(s))

    # Time-domain
    feats = [
        rms,
        kurt_stat(s),
        abs(skew_stat(s)),
        peak / (rms + 1e-8),
        np.ptp(s),
        np.mean(np.abs(s)),
        float(np.sum(s**2)),
    ]

    # Frequency-domain
    N     = len(s)
    freq  = fftfreq(N, 1/fs)[:N//2]
    amp   = np.abs(fft(s)[:N//2])
    amp_n = amp / (amp.sum() + 1e-8)   # normalised for entropy

    bf = bearing_freqs(rpm)            # dynamic frequencies for this RPM

    def band_energy(center, bw=10):
        mask = (freq >= center - bw) & (freq <= center + bw)
        return float(np.sum(amp[mask]**2))

    feats += [
        float(freq[np.argmax(amp)]),
        float(np.max(amp)),
        band_energy(bf['BPFO']),
        band_energy(bf['BPFI']),
        band_energy(bf['BSF']),
        float(np.sum(amp * freq) / (np.sum(amp) + 1e-8)),
        float(-np.sum(amp_n * np.log(amp_n + 1e-8))),
    ]
    return np.array(feats, dtype=np.float32)

print(f'Feature vector length: {len(FEATURE_NAMES)}')
print('Features:', FEATURE_NAMES)

# Quick test on one window
test_feat = extract_features(records[0]['signal'][:WINDOW], rpm=records[0]['rpm'])
print(f'\nSample feature vector from {records[0]["filename"]}:')
for name, val in zip(FEATURE_NAMES, test_feat):
    print(f'  {name:<20}: {val:.6f}')

In [ ]:
# ── Extract features for all splits ──────────────────────────────────────────
# Uses RAW (non-normalised) windows so amplitude features retain physical meaning
# RPM looked up per-segment from metadata

def extract_all_features(X_raw, meta_df):
    """Vectorised feature extraction across all windows in a split."""
    features = []
    rpms = meta_df['load_hp'].map(RPM_BY_LOAD).values
    for i in tqdm(range(len(X_raw)), desc='Extracting features', leave=False):
        features.append(extract_features(X_raw[i], fs=FS, rpm=int(rpms[i])))
    return np.array(features, dtype=np.float32)

print('Extracting features for train set...')
F_train_raw = extract_all_features(X_raw_train, meta_train)
print('Extracting features for val set...')
F_val_raw   = extract_all_features(X_raw_val, meta_val)
print('Extracting features for test set...')
F_test_raw  = extract_all_features(X_raw_test, meta_test)

# ── Scale features ────────────────────────────────────────────────────────────
# StandardScaler fit on TRAIN only, applied to val and test
# This is the correct ML practice — test set statistics must not influence scaling
scaler  = StandardScaler()
F_train = scaler.fit_transform(F_train_raw)  # fit + transform train
F_val   = scaler.transform(F_val_raw)         # transform only
F_test  = scaler.transform(F_test_raw)         # transform only

print(f'\nFeature matrix shapes:')
print(f'  F_train: {F_train.shape}  (N_segments × {len(FEATURE_NAMES)} features)')
print(f'  F_val  : {F_val.shape}')
print(f'  F_test : {F_test.shape}')
print(f'\nScaler fit on train only. Mean and std of train features after scaling:')
print(f'  Mean ≈ {F_train.mean():.4f}  (should be ~0)')
print(f'  Std  ≈ {F_train.std():.4f}   (should be ~1)')

In [ ]:
# ── Visualise: feature distributions after scaling ───────────────────────────
fig, axes = plt.subplots(2, 7, figsize=(18, 7))
for i, (ax, fname) in enumerate(zip(axes.flatten(), FEATURE_NAMES)):
    ax.hist(F_train[:, i], bins=50, color='steelblue', alpha=0.7, density=True)
    ax.set_title(fname, fontsize=8, fontweight='bold')
    ax.set_xlabel('Scaled value', fontsize=7)
    ax.tick_params(labelsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('Feature distributions after StandardScaler (train set)\n'
             'Well-scaled features have spread centred near 0',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/step6_feature_distributions.png', bbox_inches='tight')
plt.show()
print('Saved: step6_feature_distributions.png')

---
## Step 7 — Model input preparation (deep learning path)

CNN, LSTM, and Transformer models take the **raw normalised windows** directly.
No hand-crafted features — the model learns its own representations.

Two shape transformations needed:
- Add a channel dimension: `(N, 1024)` → `(N, 1024, 1)` for Keras Conv1D
- Compute class weights to handle the Normal class imbalance

In [ ]:
# ── Add channel dimension for Keras Conv1D ───────────────────────────────────
# Conv1D expects (batch, timesteps, channels)
# We have 1 channel — the accelerometer signal
X_train_dl = X_norm_train[..., np.newaxis]  # (N, 1024, 1)
X_val_dl   = X_norm_val[..., np.newaxis]
X_test_dl  = X_norm_test[..., np.newaxis]

print('Deep learning input shapes:')
print(f'  X_train_dl : {X_train_dl.shape}  (samples, timesteps, channels)')
print(f'  X_val_dl   : {X_val_dl.shape}')
print(f'  X_test_dl  : {X_test_dl.shape}')
print(f'  y_train    : {y_train.shape}  range=[{y_train.min()}, {y_train.max()}]')

In [ ]:
# ── Class weights ─────────────────────────────────────────────────────────────
# Normal class has ~3x more segments than fault classes.
# Class weights inversely proportional to frequency.
# Applied to the loss function during training so Normal does not dominate.

classes_present = np.unique(y_train)
cw_values = compute_class_weight(
    class_weight='balanced',
    classes=classes_present,
    y=y_train
)
# Build full dict for all NUM_CLASSES (some may not be in train — assign weight 1.0)
class_weight_dict = {i: 1.0 for i in range(NUM_CLASSES)}
for cls, w in zip(classes_present, cw_values):
    class_weight_dict[cls] = round(float(w), 4)

print('Class weights (train set):')
print(f'{"Class":<16} {"Label":<16} {"Count":<8} {"Weight"}')
print('-' * 52)
for cls in sorted(class_weight_dict.keys()):
    count = int(np.sum(y_train == cls))
    lbl   = ALL_LABELS[cls] if cls < len(ALL_LABELS) else '?'
    print(f'{cls:<16} {lbl:<16} {count:<8} {class_weight_dict[cls]:.4f}')

print(f'\nNormal weight  = {class_weight_dict[0]:.4f}  (downweighted — over-represented)')
print(f'Fault avg wt   = {np.mean([class_weight_dict[i] for i in range(1, NUM_CLASSES)]):.4f}  (upweighted)')

In [ ]:
# ── Visualise: normalised signal comparison across classes ─────────────────────
# Shows what the DL model actually sees as input
sample_classes = ['Normal','IR_7','IR_14','OR@6_7','OR@6_21','Ball_7','Ball_28']
t = np.arange(WINDOW) / FS * 1000

fig, axes = plt.subplots(len(sample_classes), 1, figsize=(14, 14), sharex=True)
colors = ['#607D8B','#1565C0','#42A5F5','#E65100','#FF7043','#6A1B9A','#AB47BC']

for ax, lbl, col in zip(axes, sample_classes, colors):
    cls = label2idx.get(lbl)
    if cls is None: continue
    # Find first segment from train set with this label
    idxs = np.where(y_train == cls)[0]
    if len(idxs) == 0: continue
    seg = X_train_dl[idxs[0], :, 0]
    ax.plot(t, seg, color=col, linewidth=0.7)
    ax.set_ylabel(lbl, fontsize=9)
    ax.set_ylim(-6, 6)
    ax.grid(True, alpha=0.25)
    ax.text(0.01, 0.85, f'Kurt={kurt_stat(seg):.2f}', transform=ax.transAxes, fontsize=8, color=col)

axes[-1].set_xlabel('Time (ms)')
fig.suptitle('Deep learning model input — normalised windows\n'
             '(shape preserved, amplitude standardised to zero mean / unit std)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/step7_dl_inputs.png', bbox_inches='tight')
plt.show()
print('Saved: step7_dl_inputs.png')

---
## Step 8 — Save all outputs for Notebook 4

In [ ]:
import pickle

# ── Deep learning tensors ────────────────────────────────────────────────────
np.save(f'{OUT_DIR}/X_train_dl.npy', X_train_dl)
np.save(f'{OUT_DIR}/X_val_dl.npy',   X_val_dl)
np.save(f'{OUT_DIR}/X_test_dl.npy',  X_test_dl)

# ── Classical feature matrices ────────────────────────────────────────────────
np.save(f'{OUT_DIR}/F_train.npy', F_train)
np.save(f'{OUT_DIR}/F_val.npy',   F_val)
np.save(f'{OUT_DIR}/F_test.npy',  F_test)

# ── Labels ───────────────────────────────────────────────────────────────────
np.save(f'{OUT_DIR}/y_train.npy', y_train)
np.save(f'{OUT_DIR}/y_val.npy',   y_val)
np.save(f'{OUT_DIR}/y_test.npy',  y_test)

# ── Metadata DataFrames ───────────────────────────────────────────────────────
meta_train.to_csv(f'{OUT_DIR}/meta_train.csv', index=False)
meta_val.to_csv(f'{OUT_DIR}/meta_val.csv',     index=False)
meta_test.to_csv(f'{OUT_DIR}/meta_test.csv',   index=False)

# ── Scaler (needed to transform new data at inference time) ──────────────────
with open(f'{OUT_DIR}/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# ── Class info JSON ───────────────────────────────────────────────────────────
class_info = {
    'all_labels':        ALL_LABELS,
    'label2idx':         label2idx,
    'num_classes':       NUM_CLASSES,
    'feature_names':     FEATURE_NAMES,
    'class_weight_dict': class_weight_dict,
    'window_size':       WINDOW,
    'step_size':         STEP,
    'sample_rate':       FS,
    'train_loads':       TRAIN_LOADS,
    'val_loads':         VAL_LOADS,
    'test_loads':        TEST_LOADS,
    'split_counts': {
        'train': int(len(y_train)),
        'val':   int(len(y_val)),
        'test':  int(len(y_test))
    }
}
with open(f'{OUT_DIR}/class_info.json', 'w') as f:
    json.dump(class_info, f, indent=2)

print('All outputs saved to:', OUT_DIR)
print()
for fname in sorted(os.listdir(OUT_DIR)):
    size = os.path.getsize(os.path.join(OUT_DIR, fname))
    print(f'  {fname:<30}  {size/1024:.1f} KB')

print('\nPreprocessing plots saved to:', PLOTS_DIR)
print('  step2_segmentation.png')
print('  step3_normalisation.png')
print('  step5_split_distribution.png')
print('  step6_feature_distributions.png')
print('  step7_dl_inputs.png')
print('\n✅ Preprocessing complete. Run 04_models.ipynb next.')